# Week 4: State-Level Coverage Mapping

**NWR Coverage Gap Research — Summer 2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/W2NJL/nwr-gap-research/blob/main/week4_state_sweep.ipynb)

---

In Week 3 you queried coverage for a handful of Florida cities. Now it's time to scale up: this week you'll sweep **every city in a single state**, map the results, and formally test the hypothesis that **rural communities are more likely to fall into NWR coverage gaps**.

By the end of this notebook you'll have:
- A complete coverage dataset for one US state
- A Folium map with population-scaled markers
- A quantitative gap-rate comparison across rural and urban county types
- A clear-eyed view of what the national sweep in Week 5 will require

---
## Part 1: Setup

Run these cells first. The query functions are identical to Week 3 — no changes needed.

In [ ]:
!pip install folium openpyxl --quiet

import requests, json, io, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from IPython.display import display

WX_URL     = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/wx_stations.csv"
CITIES_URL = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/us_cities.csv"
RUCC_URL   = "https://www.ers.usda.gov/webdocs/DataFiles/53251/ruralurbancontinuumcodes2023.xlsx"

wx     = pd.read_csv(WX_URL)
wx     = wx[wx['country'] == 'USA'].dropna(subset=['latitude', 'longitude']).copy()
wx['longitude'] = wx['longitude'] * -1
cities = pd.read_csv(CITIES_URL)

COVERAGE_THRESHOLD = 50.0
RADIOLAND_BASE     = "http://52.151.197.43/search_stream"

print(f"wx_stations: {len(wx)} US transmitters")
print(f"cities:      {len(cities)} US cities")

In [ ]:
def query_nwr_coverage(lat, lon, rx_height=10, min_sig_strength=3, verbose=False):
    params = {
        'lat': lat, 'lon': lon,
        'rx_height': rx_height,
        'min_sig_strength': min_sig_strength,
        'service': 'WX',
    }
    try:
        resp = requests.get(RADIOLAND_BASE, params=params, stream=True, timeout=60)
        resp.raise_for_status()
        rows = []
        for line in resp.iter_lines():
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
        if not rows:
            return None
        return pd.DataFrame(rows).sort_values('field_strength', ascending=False).reset_index(drop=True)
    except Exception as e:
        if verbose:
            print(f"  Error: {e}")
        return None


def get_best_signal(lat, lon, threshold=COVERAGE_THRESHOLD):
    df = query_nwr_coverage(lat, lon)
    if df is None or df.empty:
        return {'best_callsign': None, 'best_field_strength': 0.0,
                'best_distance_km': None, 'stations_above_threshold': 0, 'covered': False}
    best = df.iloc[0]
    return {
        'best_callsign':            best['callsign'],
        'best_field_strength':      float(best['field_strength']),
        'best_distance_km':         float(best.get('distance', np.nan)),
        'stations_above_threshold': int((df['field_strength'] >= threshold).sum()),
        'covered':                  float(best['field_strength']) >= threshold,
    }

---
## Part 2: Choosing Your State

The cell below shows how many cities each state has in our dataset. Pick a state with **50–150 cities** — large enough to reveal patterns, small enough to finish in one session.

**Good candidates:**

| State | Why it's interesting |
|-------|---------------------|
| `WV` | Mountainous terrain, largely rural — will stress-test ITM propagation |
| `WY` | Vast and sparsely populated, likely has real geographic gaps |
| `NM` | Desert terrain, large tribal land areas, diverse topography |
| `VT` | Small New England state with interesting valley/ridge terrain |
| `ND` | Flat, remote, agricultural — useful contrast to mountainous states |

You can choose any state — justify your choice in E1.

In [ ]:
city_counts = cities.groupby('state_id').size().sort_values()
print(city_counts.to_string())

In [ ]:
YOUR_STATE = 'WV'   # <-- change this to your chosen state

state_cities = cities[cities['state_id'] == YOUR_STATE].copy()
print(f"State: {YOUR_STATE}")
print(f"Cities in dataset: {len(state_cities)}")
print(f"Population range: {state_cities['population'].min():,} -- {state_cities['population'].max():,}")
print()
state_cities[['city', 'county_name', 'population']].sort_values('population', ascending=False).head(10)

### E1 -- State Selection

Before running the sweep, answer below.

1. Why did you choose this state? What do you expect to find?
2. Glance at the city list. Which city do you predict will have the worst coverage, and why?
3. The `population` column tells you who lives in cities, but not who is *vulnerable*. What other data would you want to layer on to measure true emergency risk?

**Your answers:**

1.
2.
3.

---
## Part 3: The State Sweep

The cell below runs `get_best_signal()` for every city in your chosen state. Each query takes 15--30 seconds, so **this will run for a while -- don't close the tab**.

A few notes:
- A 2-second sleep between calls is polite to the API and reduces timeouts.
- Results are saved to a CSV when done so you can reload without re-querying.

In [ ]:
results = []

for i, row in state_cities.reset_index(drop=True).iterrows():
    label = f"{row['city']}, {YOUR_STATE}"
    print(f"[{i+1}/{len(state_cities)}] {label}...", end=' ', flush=True)

    summary = get_best_signal(row['lat'], row['lng'])
    summary.update({
        'city':        row['city'],
        'state_id':    row['state_id'],
        'lat':         row['lat'],
        'lng':         row['lng'],
        'population':  row['population'],
        'county_name': row.get('county_name', ''),
    })
    results.append(summary)

    sig    = summary['best_field_strength']
    status = 'COVERED' if summary['covered'] else 'GAP'
    print(f"{status} ({sig:.1f} dB)")
    time.sleep(2)

coverage_df = pd.DataFrame(results)
print(f"\nDone. {coverage_df['covered'].sum()} covered, {(~coverage_df['covered']).sum()} gaps.")
coverage_df.to_csv(f"{YOUR_STATE}_coverage.csv", index=False)
print(f"Saved to {YOUR_STATE}_coverage.csv")

In [ ]:
# If you need to reload results without re-running the sweep, uncomment:
# coverage_df = pd.read_csv(f"{YOUR_STATE}_coverage.csv")
# print(f"Loaded {len(coverage_df)} rows.")

---
## Part 4: Coverage Map

### E2 -- Build the Coverage Map

Build a Folium map showing all surveyed cities. Requirements:
- **Green circles** for covered cities (field strength >= 50 dB)
- **Red circles** for gap cities (< 50 dB)
- **Circle radius** proportional to population -- try `np.log1p(row['population']) / 2`
- **Popup** on each marker: city name, best signal (dB), best station callsign, covered/gap status

Center the map on your state's average lat/lng.

In [ ]:
# YOUR CODE HERE

m = folium.Map(
    location=[state_cities['lat'].mean(), state_cities['lng'].mean()],
    zoom_start=7,
    tiles='CartoDB positron'
)

for _, row in coverage_df.iterrows():
    color  = 'green' if row['covered'] else 'red'
    radius = np.log1p(row['population']) / 2
    popup  = folium.Popup(
        f"<b>{row['city']}</b><br>"
        f"Signal: {row['best_field_strength']:.1f} dB<br>"
        f"Station: {row['best_callsign']}<br>"
        f"{'COVERED' if row['covered'] else 'GAP'}",
        max_width=200
    )
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=radius,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=popup
    ).add_to(m)

display(m)

**Map Questions -- answer below:**

1. Are gap cities clustered in a particular region of the state, or scattered throughout?
2. Do gap circles tend to be smaller (less populated) than covered ones?
3. Name one city that surprised you -- either covered when you expected a gap, or vice versa. What might explain it?

**Your answers:**

1.
2.
3.

---
## Part 5: Gap Analysis

### E3 -- Coverage Statistics

In [ ]:
total_cities   = len(coverage_df)
covered_cities = int(coverage_df['covered'].sum())
gap_cities     = total_cities - covered_cities
total_pop      = coverage_df['population'].sum()
covered_pop    = coverage_df.loc[ coverage_df['covered'],  'population'].sum()
gap_pop        = coverage_df.loc[~coverage_df['covered'], 'population'].sum()

print(f"State: {YOUR_STATE}")
print(f"{'Cities surveyed:':<32} {total_cities}")
print(f"{'Covered:':<32} {covered_cities}  ({covered_cities/total_cities:.1%})")
print(f"{'Gaps:':<32} {gap_cities}  ({gap_cities/total_cities:.1%})")
print()
print(f"{'Total population surveyed:':<32} {total_pop:,}")
print(f"{'Population with coverage:':<32} {covered_pop:,}  ({covered_pop/total_pop:.1%})")
print(f"{'Population in gaps:':<32} {gap_pop:,}  ({gap_pop/total_pop:.1%})")

### E4 -- Top Gap Cities by Population

List the 10 most-populated cities in your state that fall into coverage gaps.

In [ ]:
top_gaps = (coverage_df[~coverage_df['covered']]
           .nlargest(10, 'population')
           [['city', 'population', 'best_field_strength', 'best_callsign', 'best_distance_km']])

print(f"Top 10 gap cities in {YOUR_STATE} by population:")
print(top_gaps.to_string(index=False))

**Questions:**

1. What is the largest (by population) gap city? What's its best field strength?
2. Is distance a reliable predictor of gaps, or are some cities close to a transmitter but still below threshold?
3. Pick one gap city, look it up: what county is it in, and what natural hazards does that area face? (Draw on Week 2.)

**Your answers:**

1.
2.
3.

---
## Part 6: The Rural Hypothesis

In Week 2 you inventoried the **USDA Rural-Urban Continuum Codes (RUCC)**. The central hypothesis of this research is: *rural communities are more likely to fall into NWR coverage gaps.*

Week 3 let you preview this join with a 10-city sample. Now test it against the full state sweep.

**RUCC scale:**

| Code | Classification |
|------|---------------|
| 1--3 | Metro counties |
| 4--6 | Non-metro, adjacent to metro |
| 7--9 | Non-metro, not adjacent (most rural) |

In [ ]:
import urllib.request

urllib.request.urlretrieve(RUCC_URL, "rucc2023.xlsx")
rucc = pd.read_excel("rucc2023.xlsx")
rucc.columns = rucc.columns.str.strip()
print("Columns:", rucc.columns.tolist())
print(f"Rows: {len(rucc)}")
rucc.head(3)

In [ ]:
rucc['county_clean'] = (rucc['County_Name']
                         .str.replace(r' County$', '', regex=True)
                         .str.strip().str.upper())

coverage_df['county_upper'] = coverage_df['county_name'].str.upper().str.strip()

rucc_state = (rucc[rucc['State'] == YOUR_STATE]
              [['county_clean', 'RUCC_2023']]
              .drop_duplicates())

coverage_rucc = coverage_df.merge(
    rucc_state,
    left_on='county_upper',
    right_on='county_clean',
    how='left'
)

matched = coverage_rucc['RUCC_2023'].notna().sum()
print(f"Counties matched: {matched} / {len(coverage_rucc)}")
if matched < len(coverage_rucc) * 0.8:
    print("Warning: fewer than 80% matched -- check county_name values in your cities data")

### E5 -- Metro vs. Rural Gap Rates

Compare coverage gap rates between metro and rural counties.

In [ ]:
# YOUR CODE HERE
# 1. Create boolean column 'metro' -- True for RUCC 1-3, False for 4-9
# 2. Group by 'metro', compute: city count, gap count, gap rate, avg field strength
# 3. Print the comparison

coverage_rucc['metro'] = coverage_rucc['RUCC_2023'] <= 3

summary = (coverage_rucc
           .groupby('metro')
           .agg(
               city_count = ('city', 'count'),
               gap_count  = ('covered', lambda x: (~x).sum()),
               avg_signal = ('best_field_strength', 'mean'),
           )
           .assign(gap_rate=lambda d: d['gap_count'] / d['city_count']))

summary.index = summary.index.map({True: 'Metro (RUCC 1-3)', False: 'Non-Metro (RUCC 4-9)'})
print(summary.to_string())

**Questions:**

1. What is the gap rate for metro vs. non-metro counties?
2. Is the difference large enough to be meaningful given your sample size, or could it be noise?
3. What factors *besides rurality* could explain coverage differences? Think terrain, transmitter siting history, and ERP levels.

**Your answers:**

1.
2.
3.

---
## Part 7: Signal Strength Distribution

### E6 -- Histogram

A coverage map shows *where* gaps are. A histogram shows *how bad* they are. A city receiving 48 dB is very different from one at 10 dB -- both are "gaps," but the first might be fixable with a modest power increase while the second may need a new transmitter entirely.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

covered_sig = coverage_df.loc[ coverage_df['covered'],  'best_field_strength']
gap_sig     = coverage_df.loc[~coverage_df['covered'], 'best_field_strength']

bins = np.linspace(0, coverage_df['best_field_strength'].max() + 5, 30)
ax.hist(covered_sig, bins=bins, color='steelblue', alpha=0.8, label='Covered')
ax.hist(gap_sig,     bins=bins, color='tomato',    alpha=0.8, label='Gap')
ax.axvline(COVERAGE_THRESHOLD, color='black', linestyle='--', linewidth=1.5,
           label=f'Threshold ({COVERAGE_THRESHOLD} dB)')

ax.set_xlabel('Best Field Strength (dB µV/m)')
ax.set_ylabel('Number of Cities')
ax.set_title(f'NWR Signal Strength Distribution -- {YOUR_STATE}')
ax.legend()
plt.tight_layout()
plt.show()

**Questions:**

1. Are gap cities clustered near the threshold (borderline) or far below it (deep gaps)?
2. What does the shape of the covered distribution tell you -- are most covered cities well above threshold or barely over it?
3. If you could place one new NWR transmitter anywhere in this state, where would you put it and why?

**Your answers:**

1.
2.
3.

---
## Week 4 Reflection

1. A city-level sweep counts places but misses people who live *outside* city limits -- farms, rural communities, unincorporated areas. How significant is that blind spot for this research?
2. Did the rural hypothesis hold for your state? What additional evidence would it take to convince you the pattern is real, not just geographic coincidence?
3. In Week 5 you'll scale to the full United States. Describe your sampling strategy -- will you query every city, filter by population, or sample per state? What tradeoffs does your approach make?
4. The 50 dB threshold is an engineering assumption built into the RadioLand model, not a regulatory standard. What would it take to establish a formal threshold -- and who should have the authority to set it?

**Your answers:**

1.
2.
3.
4.